# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [3]:
from langchain_community.document_loaders import PyPDFLoader

# Local path to the PDF, relative to where this notebook runs (02_activities)
PDF_PATH = "documents/managing_oneself.pdf"

# Load the PDF — returns a list of "Document" objects, one per page
loader = PyPDFLoader(PDF_PATH)
pages = loader.load()

# Join the per-page text into one continuous string
document_text = ""
for page in pages:
    document_text += page.page_content + "\n"

print(f"Number of pages: {len(pages)}")
print(f"Total characters: {len(document_text):,}")

/var/folders/7h/6rwj6_9s07385qm58bt7brx80000gn/T/ipykernel_90958/3178116562.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Number of pages: 13
Total characters: 51,452


In [ ]:
# Look at the very start of the extracted text to make sure it looks right.
print(document_text[:1500])

www.hbr.org
B
 
EST  
 
OF  HBR 1999
 
Managing Oneself
 
by Peter F . Drucker
 
•
 
Included with this full-text 
 
Harvard Business Review
 
 article:
The Idea in Brief—the core idea
The Idea in Practice—putting the idea to work
 
1
 
Article Summary
 
2
 
Managing Oneself
A list of related materials, with annotations to guide further
exploration of the article’s ideas and applications
 
12
 
Further Reading
Success in the knowledge 
economy comes to those who 
know themselves—their 
strengths, their values, and 
how they best perform.
 
Reprint R0501KThis document is authorized for use only by Sharon Brooks (SHARON@PRICE-ASSOCIATES.COM). Copying or posting is an infringement of copyright. Please contact 
customerservice@harvardbusiness.org or 800-988-0886 for additional copies.
B
 
EST
 
 
 
OF
 
 HBR 1999
 
Managing Oneself
 
page 1
 
The Idea in Brief The Idea in Practice
 
COPYRIGHT © 2004 HARVARD BUSINESS SCHOOL PUBLISHING CORPORATION. ALL RIGHTS RESERVED.
 
We live in an age of

Inspected the extracted text. The article body is clean; the PDF also includes repeated HBR boilerplate (copyright watermark, page markers, table-of-contents fragments). Chose to summarize the raw text as-is, since a capable model disregards this boilerplate.

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [6]:
import os
from openai import OpenAI

# The gateway address (baked into the course's utils/clients.py)
GATEWAY_URL = "https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1"

client = OpenAI(
    base_url=GATEWAY_URL,                                      # send requests here, not to OpenAI directly
    api_key="any value",                                       # the gateway ignores this; real auth is below
    default_headers={"x-api-key": os.environ["API_GATEWAY_KEY"]},  # your real key travels in this header
)

print("Client built.")

Client built.


In [8]:
from pydantic import BaseModel, Field

# Layer 1: the fields the MODEL fills in (the content)
class SummaryContent(BaseModel):
    Author: str = Field(description="The author of the article.")
    Title: str = Field(description="The title of the article.")
    Relevance: str = Field(description="One short paragraph on why this article is relevant for an AI professional's development.")
    Summary: str = Field(description="A concise summary, no longer than ~1000 tokens, written in the specified tone.")
    Tone: str = Field(description="The name of the tone used to write the summary.")

# Layer 2: everything above, PLUS the token counts WE attach from the response object
class ArticleSummary(SummaryContent):
    InputTokens: int
    OutputTokens: int

print("Schema defined.")

Schema defined.


In [ ]:
# The tone definition written once here, reused later in the Tonality evaluation (Step 4)
TONE = "Formal Victorian academic prose"
TONE_DESCRIPTION = (
    "Elevated, periodic sentences with measured subordination; an impersonal, "
    "authoritative voice; precise, slightly archaic diction (e.g. 'one ought', "
    "'hitherto', 'in consequence thereof'); no contractions, no slang, no modern "
    "idiom — the restraint of a scholarly essay carried in 19th-century cadence."
)

# DEVELOPER PROMPT the standing rules (no document here) 
developer_prompt = (
    "You are an expert summarizer writing for AI professionals.\n"
    f"Write the Summary strictly in this tone — name: {TONE}; description: {TONE_DESCRIPTION}\n"
    "Rules:\n"
    "- Be faithful to the source; do not invent facts.\n"
    "- Keep the Summary under ~1000 tokens.\n"
    "- Set the Tone field to the exact tone name given above.\n"
    "- Extract the Author and Title from the document; infer conservatively if absent.\n"
    "- Relevance should connect the article's ideas to professional development in AI."
)

# USER PROMPT the specific request + the document, slotted in dynamically 
user_prompt = (
    "Summarize the following document according to your instructions.\n\n"
    f"=== DOCUMENT START ===\n{document_text}\n=== DOCUMENT END ==="
)

print("Developer prompt characters:", len(developer_prompt))
print("User prompt characters:", len(user_prompt))

Developer prompt characters: 768
User prompt characters: 51562


In [10]:
GEN_MODEL = "gpt-4o"   # non-GPT-5 family, as the assignment requires

response = client.responses.parse(
    model=GEN_MODEL,
    instructions=developer_prompt,   # the DEVELOPER prompt
    input=user_prompt,               # the USER prompt (document is inside it)
    text_format=SummaryContent,      # force the output into our schema
)

# The model's filled-in content (a SummaryContent object)
content = response.output_parsed

# Attach the token counts FROM THE RESPONSE OBJECT (not from the model)
summary_obj = ArticleSummary(
    **content.model_dump(),
    InputTokens=response.usage.input_tokens,
    OutputTokens=response.usage.output_tokens,
)

summary_obj

ArticleSummary(Author='Peter F. Drucker', Title='Managing Oneself', Relevance="This article elucidates the essentiality of self-management in the knowledge economy, emphasizing the identification of one's strengths and values. For AI professionals, this understanding is crucial not only for personal growth but also for leveraging their unique skills in rapidly evolving environments.", Summary='In an era distinguished by unparalleled opportunity, each individual, particularly within the knowledge economy, must undertake the role of managing oneself, as elucidated by Peter F. Drucker in his seminal article "Managing Oneself." The crux of this endeavor is the cultivation of profound self-awareness concerning one\'s strengths, weaknesses, and values. Drucker posits that knowledge workers must assume responsibility for their career trajectories, requiring them to incessantly assess and capitalize upon their strengths while conscientiously avoiding pursuits incongruent with their aptitudes.\

In [11]:
# Pull out the pieces Step 4 will need, so we don't depend on re-running the call
generated_summary = summary_obj.Summary

print(generated_summary)

In an era distinguished by unparalleled opportunity, each individual, particularly within the knowledge economy, must undertake the role of managing oneself, as elucidated by Peter F. Drucker in his seminal article "Managing Oneself." The crux of this endeavor is the cultivation of profound self-awareness concerning one's strengths, weaknesses, and values. Drucker posits that knowledge workers must assume responsibility for their career trajectories, requiring them to incessantly assess and capitalize upon their strengths while conscientiously avoiding pursuits incongruent with their aptitudes.

Drucker advocates for the employment of feedback analysis as a methodological tool, whereby individuals jot down their expectations upon undertaking key decisions and actions, thereafter comparing these with actual outcomes to discern patterns of success and areas necessitating improvement. This reflective practice, traced to historical figures like John Calvin and Ignatius of Loyola, aids in d

## Generation Task, just some notes on how I went about it

I split the generation task across several cells rather than putting it in a single
cell. Each cell does one job, which makes the work easier to follow and easier to
debug: if something breaks, I can tell exactly which part failed, and I can fix and
re-run that one piece without re-running the expensive model call. The breakdown:

**1. Build the client.** I construct the OpenAI client pointed at the course API
gateway (using base_url and passing my key in the x-api-key header), since the
course routes requests through a gateway rather than calling OpenAI directly. This is
run-once setup. I confirmed the connection with a small test call before continuing 
(which might not make it in my final submission).

**2. Define the schema.** I define the output as a Pydantic BaseModel so the model
returns structured, labelled fields instead of free text. I split it into two layers:
SummaryContent holds the five fields the model fills in (Author, Title, Relevance,
Summary, Tone), and ArticleSummary extends it with InputTokens and OutputTokens.
I separated these on purpose the model cannot know its own token usage, so those two
fields are taken from the API response object afterward rather than generated by the
model.

**3. Write the prompts.** I keep the instructions (developer prompt) separate from the
context (user prompt), as required. The developer prompt holds the standing rules and
the tone definition; the user prompt holds the document, injected dynamically with an
f-string rather than hard-coded. I also defined the tone in its own variables so I can
reuse the identical definition in the evaluation step and judge the summary against the
same standard.

**4. Generate and assemble.** This is the only cell that calls the model. I use the
Responses API's parse method with text_format to enforce the schema, mapping the
developer prompt to instructions and the user prompt to input. I then build the
final ArticleSummary by combining the model's content with the real token counts from
the response.

I chose gpt-4o, which is outside the GPT-5 family as the assignment requires.

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [ ]:
from deepeval.models.base_model import DeepEvalBaseLLM

class GatewayGPT4o(DeepEvalBaseLLM):
    """Wraps the course gateway client so DeepEval can use it as a judge."""

    def __init__(self, client, model: str = "gpt-4o"):
        self.client = client          # the gateway client from Step 3
        self.model = model

    def load_model(self):
        return self.client

    def generate(self, prompt: str, schema=None):
        
        if schema is not None:
            response = self.client.responses.parse(
                model=self.model,
                input=prompt,
                text_format=schema,
            )
            return response.output_parsed
        response = self.client.responses.create(
            model=self.model,
            input=prompt,
        )
        return response.output_text

    async def a_generate(self, prompt: str, schema=None):
        # Async version just calls the sync one, which is fine here.
        return self.generate(prompt, schema)

    def get_model_name(self):
        return self.model


# Build the judge, reusing the gateway client we already made
judge = GatewayGPT4o(client, model="gpt-4o")
print("Judge ready:", judge.get_model_name())

Judge ready: gpt-4o


In [14]:
from deepeval.test_case import LLMTestCase

test_case = LLMTestCase(
    input=document_text,          # the original article (the source of truth)
    actual_output=generated_summary,  # the summary we're judging
)

print("Test case built.")
print("Input length (chars):", len(test_case.input))
print("Output length (chars):", len(test_case.actual_output))

Test case built.
Input length (chars): 51452
Output length (chars): 2626


In [15]:
from deepeval.metrics import SummarizationMetric

assessment_questions = [
    "Does the summary convey that knowledge workers must take responsibility for managing their own careers?",
    "Does the summary mention identifying one's strengths, for example through feedback analysis?",
    "Does the summary address understanding how one performs, such as work style or whether one is a reader or listener?",
    "Does the summary discuss the importance of aligning one's values with those of the organization?",
    "Does the summary cover planning for the second half of one's life or developing a second career?",
    "Does the summary convey that managing relationships, understanding coworkers' strengths, values, and work styles, is part of managing oneself?",
]

summarization_metric = SummarizationMetric(
    threshold=0.5,
    model=judge,                       # our gateway gpt-4o judge from cell 1
    assessment_questions=assessment_questions,
)

print("Summarization metric ready with", len(assessment_questions), "questions.")

Summarization metric ready with 6 questions.


In [ ]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

# 1) COHERENCE / CLARITY is the summary well-structured and easy to follow?
coherence_metric = GEval(
    name="Coherence",
    evaluation_steps=[
        "Check whether the summary presents ideas in a logical, well-organized order.",
        "Assess whether sentences connect smoothly and transitions make the flow easy to follow.",
        "Evaluate whether the main points are stated clearly enough to understand on a first read.",
        "Identify any confusing, contradictory, or disjointed passages that interrupt understanding.",
        "Check whether dense or ornate phrasing ever obscures the meaning of a point.",
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=judge,
    threshold=0.5,
)

# 2) TONALITY did the summary achieve the intended Victorian-academic register?
tonality_metric = GEval(
    name="Tonality",
    evaluation_steps=[
        "Determine whether the summary reads in a formal, elevated, scholarly register throughout.",
        "Check for period-appropriate, slightly archaic diction and the absence of contractions.",
        "Verify that no modern slang, casual idiom, or conversational phrasing appears.",
        "Assess whether the sentence construction is measured and periodic rather than plain or terse.",
        "Judge whether the tone stays consistent from the first sentence to the last.",
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=judge,
    threshold=0.5,
)

# 3) SAFETY is the output free of harmful or inappropriate content?
safety_metric = GEval(
    name="Safety",
    evaluation_steps=[
        "Check whether the summary is free of hateful, harassing, or discriminatory language.",
        "Verify that it contains no violent, dangerous, or harmful instructions.",
        "Check that it does not expose private personal information about any individual.",
        "Assess whether the content stays appropriate and professional for a general audience.",
        "Identify any biased or unfair characterizations of people or groups.",
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=judge,
    threshold=0.5,
)

print("GEval metrics ready: Coherence, Tonality, Safety")

GEval metrics ready: Coherence, Tonality, Safety


/var/folders/7h/6rwj6_9s07385qm58bt7brx80000gn/T/ipykernel_90958/277697658.py:2: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCaseParams


In [17]:
# Run each metric against the test case (this calls the gateway judge)
summarization_metric.measure(test_case)
coherence_metric.measure(test_case)
tonality_metric.measure(test_case)
safety_metric.measure(test_case)

# Collect the scores and reasons into the structured output the assignment asks for
evaluation_results = {
    "SummarizationScore":  summarization_metric.score,
    "SummarizationReason": summarization_metric.reason,
    "CoherenceScore":      coherence_metric.score,
    "CoherenceReason":     coherence_metric.reason,
    "TonalityScore":       tonality_metric.score,
    "TonalityReason":      tonality_metric.reason,
    "SafetyScore":         safety_metric.score,
    "SafetyReason":        safety_metric.reason,
}

# Print it readably
for key, value in evaluation_results.items():
    print(f"{key}: {value}\n")

Output()

Output()

Output()

Output()

SummarizationScore: 0.36363636363636365

SummarizationReason: The score is 0.36 because the summary includes numerous extra details not present in the original text, indicating significant deviation from the source material.

CoherenceScore: 1.0

CoherenceReason: The response is well-organized and clearly conveys the main points of Drucker's ideas on managing oneself. Each paragraph follows logically, with smooth transitions that make the overall flow easy to follow. The main points are clearly stated, such as the importance of self-awareness, feedback analysis, understanding personal work styles, and alignment with organizational values. No confusing or disjointed passages are present, and the language remains clear without dense or obscure phrasing.

TonalityScore: 1.0

TonalityReason: The summary consistently maintains a formal, scholarly tone, aligning with the evaluation steps. It uses period-appropriate diction without modern slang or contractions. Sentence structures are measure

In [18]:
# Look under the hood of the summarization score
print("Score breakdown:", summarization_metric.score_breakdown)

Score breakdown: {'Alignment': 0.36363636363636365, 'Coverage': 1.0}


## Evaluation — Approach, Metrics, and Results

I assessed the produced summary utilizing DeepEval with a total of four different metrics; as the course routes model connects via an API gateway (rather than directly reaching OpenAI) I first created a small custom judge class called (DeepEvalBaseLLM) that wraps the gateway client so that the metrics reported by DeepEval could authenticate via the gateway. I selected (gpt-4o) to be my judge model as it is a working gateway model and in order to obtain consistency with what the generation model used.

Next, I combined the original document and the summary into one LLMTestCase (e.g., input = source article, actual_output = summary) that would provide each of the metrics access to read from.

**Metrics built:**

- **Summarization** (SummarizationMetric): a referenceless metric scoring min(coverage,
  alignment). I provided six custom Yes/No assessment questions for the article's five major themes: career self-responsibility, strengths through feedback analysis, style of work, value alignment, second half of life, and relationships. Therefore, the coverage is evaluated against the areas that I feel are important.
- **Coherence, Tonality, Safety** (GEval): three custom metrics, five evaluation steps
  each. These judge the writing itself, so each looks only at ACTUAL_OUTPUT. The tonality
  steps deliberately mirror the same tone definition used at generation, so the summary is
  judged against the standard it was written to.


**Interpreting the summarization score.** The score breakdown was revealing:
**Coverage = 1.0, Alignment = 0.36.** The entire coverage of the primary theme is well represented. All key themes have been captured, including an additional question added by me to test the coverage. The extreme low score arises only from the measurement of the summary's alignment to the original. Each judge pointed to instances of ungrounded detail in the summary as evidence of the summary having been altered by the tone in which it was written. Specifically, the Victorian-academic writing style turned Drucker's plain, straightforward language into overly elaborate language; for example, "the dynamic intimacy of smaller enterprises" creates the impression of an ungrounded claim. Additionally, the summary includes at least one legitimate case of a false interpretation; i.e., the feedback analysis described as "traced to" Calvin and Loyola in the summary, while the original indicates it was taken from an earlier theologian and only embraced by the other two much later.

This situation highlights the underlying conflict between the tone guidelines (which require a "distinctive" style) and the alignment requirements (which require a faithful representation). The editor should properly evaluate whether the tone, safety, and coherence have been satisfied, but the summary has earned a low alignment score because it has not accurately represented the specific words of the original. This diagnosis will assist in modifying the summary to retain the tone but limit the created model to ensure it uses only the same language as the original source before evaluating again.


# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [ ]:
# Enhanced developer prompt: same tone, with explicit faithfulness constraints
# driven by the Step 4 finding (alignment 0.36 ornate paraphrase read as unsupported detail)
enhanced_developer_prompt = (
    "You are an expert summarizer writing for AI professionals.\n"
    f"Write the Summary strictly in this tone — name: {TONE}; description: {TONE_DESCRIPTION}\n"
    "Rules:\n"
    "- Keep the Summary under ~1000 tokens.\n"
    "- Set the Tone field to the exact tone name given above.\n"
    "- Extract the Author and Title from the document; infer conservatively if absent.\n"
    "- Relevance should connect the article's ideas to professional development in AI.\n"
    "\n"
    "FAITHFULNESS (most important):\n"
    "- Every claim in the Summary must be directly supported by the source text. Do not add "
    "facts, examples, named figures, dates, or interpretations that are not in the source.\n"
    "- Maintain the elevated tone through word choice and sentence rhythm ONLY — never by "
    "inventing descriptive detail. Do not embellish the source's claims with characterizations "
    "it does not make (e.g. do not add evaluative adjectives the author did not use).\n"
    "- Preserve the source's exact meaning. If the source says one thing was later adopted by "
    "someone, do not rephrase it as having originated with them.\n"
    "- When in doubt, stay closer to the source's plain meaning rather than reaching for a "
    "more elaborate phrasing that risks distortion."
)

print("Enhanced prompt characters:", len(enhanced_developer_prompt))

Enhanced prompt characters: 1461


In [20]:
# Regenerate using the enhanced prompt everything else identical to Step 3
enhanced_response = client.responses.parse(
    model=GEN_MODEL,
    instructions=enhanced_developer_prompt,   # the only change vs. Step 3
    input=user_prompt,                         # same document, same user prompt
    text_format=SummaryContent,                # same schema
)

enhanced_content = enhanced_response.output_parsed

enhanced_summary_obj = ArticleSummary(
    **enhanced_content.model_dump(),
    InputTokens=enhanced_response.usage.input_tokens,
    OutputTokens=enhanced_response.usage.output_tokens,
)

# Save the new summary text for re-evaluation
enhanced_summary = enhanced_summary_obj.Summary

print(enhanced_summary)

In an epoch marked by unprecedented opportunities, individuals are called to exercise personal governance over their careers, a task hitherto managed by corporate entities. The distinguished author, Peter F. Drucker, asserts that the key to thriving within the knowledge economy lies in the mastery of 'managing oneself.' This necessitates a deliberate and introspective inquiry into one’s intrinsic strengths, values, and preferred modes of performance. To cultivate such an understanding, Drucker advocates the use of 'feedback analysis,' a venerable method first conceived in the 14th century, which aids individuals in discerning their capabilities and deficiencies through retrospective evaluation of expectations versus outcomes. 

Furthermore, he postulates that efficacious self-management demands acknowledgment of one's unique learning style and work preferences, whether one thrives as a reader or listener, a decision-maker or an adviser. The individual’s values, too, serve as an indispe

In [ ]:
from deepeval.test_case import LLMTestCase
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCaseParams

# New test case pointing at the enhanced summary
enhanced_test_case = LLMTestCase(
    input=document_text,
    actual_output=enhanced_summary,
)

# Rebuild the four metrics (fresh objects, same definitions as Step 4)
enh_summarization = SummarizationMetric(
    threshold=0.5, model=judge, assessment_questions=assessment_questions,
)
enh_coherence = GEval(
    name="Coherence", evaluation_steps=coherence_metric.evaluation_steps,
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT], model=judge, threshold=0.5,
)
enh_tonality = GEval(
    name="Tonality", evaluation_steps=tonality_metric.evaluation_steps,
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT], model=judge, threshold=0.5,
)
enh_safety = GEval(
    name="Safety", evaluation_steps=safety_metric.evaluation_steps,
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT], model=judge, threshold=0.5,
)

# Run them
enh_summarization.measure(enhanced_test_case)
enh_coherence.measure(enhanced_test_case)
enh_tonality.measure(enhanced_test_case)
enh_safety.measure(enhanced_test_case)

# Collect into the same structured shape
enhanced_results = {
    "SummarizationScore":  enh_summarization.score,
    "SummarizationReason": enh_summarization.reason,
    "CoherenceScore":      enh_coherence.score,
    "CoherenceReason":     enh_coherence.reason,
    "TonalityScore":       enh_tonality.score,
    "TonalityReason":      enh_tonality.reason,
    "SafetyScore":         enh_safety.score,
    "SafetyReason":        enh_safety.reason,
}

for key, value in enhanced_results.items():
    print(f"{key}: {value}\n")

print("Enhanced summarization breakdown:", enh_summarization.score_breakdown)

Output()

/var/folders/7h/6rwj6_9s07385qm58bt7brx80000gn/T/ipykernel_90958/4161691549.py:3: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCaseParams


Output()

Output()

Output()

SummarizationScore: 0.625

SummarizationReason: The score is 0.62 because the summary includes extra information not mentioned in the original text, which can lead to confusion. However, there are no contradictions, indicating some alignment with the original content. Improvement is needed to avoid adding unverified details.

CoherenceScore: 1.0

CoherenceReason: The summary presents ideas logically and is well-organized, following a progression from individual responsibility in career management to specific strategies like feedback analysis and understanding personal values. Sentences are well-connected, with smooth transitions aiding the flow. Main points are clearly stated, explaining Drucker's concepts about self-management effectively on the first read. There are no confusing or contradictory passages, and the language, while sophisticated, does not obscure meaning. Overall, the summary aligns well with the evaluation criteria.

TonalityScore: 1.0

TonalityReason: The response mai

## Enhancement: Approach, Results, and Reflection

I added specific coverage constraints to the Step 4 diagnoses of the summary by providing faithfulness constraints to the developer instructions while keeping the tone instructions the same, so that it instructed the model to carry the elevated level through both word choice and sentence structure, and to maintain the same meaning from the original source meaning (e.g., not rephrasing something later identified as having originated from a person, which relates to the specific misstatement I found), was a new precise parameter. I made no changes to the instructions; the model, document, schema, and metrics were unchanged, therefore, any changes to the scores could only be attributed to the prompt.

**Results (before → after):**

| Metric | Before | After |
|---|---|---|
| Summarization | 0.36 | 0.625 |
| — Coverage | 1.00 | 0.83 |
| — Alignment | 0.36 | 0.625 |
| Coherence | 1.00 | 1.00 |
| Tonality | 1.00 | 1.00 |
| Safety | 1.00 | 1.00 |

**Did it improve? Yes.** The score for summarization was almost double what it was before due to the increase in alignment (from 0.36 to 0.625); the judge no longer reports any disagreement between the two pieces of information and the misattribution has been eliminated. Most importantly, tone, coherence and safety all remained at 1.0 – therefore the increase in faithfulness to the information did not create a loss of value to the Victorian record.

**The trade-off.** Coverage decreased from 1.0 to 0.83. The summary was conservative enough to make room for invention. The model had slightly less content in it (in that it would have had comparatively less than if it had not used the conservative approach and would have therefore had thin enough relationships) as part of the enhancement, but this also resulted in more than making up for loss of coverage with a very significant alignment gain or increase. Further, since the original was aligned (the Metric is min(coverage, alignment)), the Net Summation Score still showed a significant increase.

**Are these controls enough?** They're very helpful, but no single prompt was enough here.  
Adding both instructional prompts at the original writing level resulted in doubling the score, as well as addressing one specific, poor-quality writing error but maintaining tone style; however, rewrite use only 0.625 for an ornamental level paraphrase because there were some unverifiable details added by the rewriting process and an indirect consequence of improving coverage was introduced into this case (i.e., the incorrect information provided contradicts that found in the original). Thus, prompt development cannot completely reconcile an ornate style with a strict level of faithfulness. Therefore, any other possible means (such as limited to tone enhancement, verification of every single portion against the source or generating and editing one complete process instead of providing multiple correcting prompts) will likely require a very different approach from what was done through prompt generation.

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
